In [57]:
# ==============================================================================
# DATA PREPARATION AND ENRICHMENT
# ==============================================================================


In [58]:
# Increase SWAP Memory on Ubuntu
# sudo swapoff -a
# sudo dd if=/dev/zero of=/swapfile bs=1M count=19456
# sudo chmod 600 /swapfile
# sudo mkswap /swapfile
# sudo swapon /swapfile

In [59]:
# ==============================================================================
# IMPORTS
# ==============================================================================


In [60]:

import gc
import hashlib
from collections import Counter, defaultdict
import json
import os
import re
import sys
import time
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


In [61]:

# ==============================================================================
# CONFIG
# ==============================================================================


In [62]:

SEED = 42
SAVE_TAG = "refreshed_v3_memorysafe"
YEAR_FILTER = 2023

PROJECT_ROOT = Path("..")
RAW_DIR = PROJECT_ROOT / "data" / "unprocessed"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

SISREG_FILE = RAW_DIR / "sisreg.csv"
CEP_FILE = RAW_DIR / "cepbrazil.csv"

TARGET_COL = "ausencia"

# Memory-safe CEP processing. The CEP lookup can be very large, so this cell never loads
# the full cepbrazil.csv into memory. It reads only the required columns in chunks,
# keeps exact rows only for CEPs present in the SISREG data, and builds prefix
# recovery only for CEPs still unmatched after the exact lookup.
CEP_CHUNK_SIZE = 500_000
CEP_PREFIX_MIN_LEN = 5
MAX_CEP_CONFLICT_REPORT_ROWS = 50_000


# Files created by this refreshed version. All have SAVE_TAG in the filename.
OUTPUT_PATHS = {
    "dados_uteis_pkl": PROCESSED_DIR / f"dados_uteis_{SAVE_TAG}.pkl",
    "dados_uteis_sample_csv": PROCESSED_DIR / f"dados_uteis_sample_{SAVE_TAG}.csv",
    "pipeline_config_json": PROCESSED_DIR / f"pipeline_config_{SAVE_TAG}.json",
    "quality_initial_csv": PROCESSED_DIR / f"data_quality_audit_initial_raw_{SAVE_TAG}.csv",
    "quality_after_basic_csv": PROCESSED_DIR / f"data_quality_audit_after_basic_cleaning_{SAVE_TAG}.csv",
    "quality_pre_final_csv": PROCESSED_DIR / f"data_quality_audit_pre_final_{SAVE_TAG}.csv",
    "quality_final_csv": PROCESSED_DIR / f"data_quality_audit_final_saved_{SAVE_TAG}.csv",
    "row_drop_summary_csv": PROCESSED_DIR / f"row_drop_summary_{SAVE_TAG}.csv",
    "class_distribution_csv": PROCESSED_DIR / f"class_distribution_{SAVE_TAG}.csv",
    "cep_lookup_clean_pkl": PROCESSED_DIR / f"cepbrazil_clean_lookup_{SAVE_TAG}.pkl",
    "cep_conflicts_csv": PROCESSED_DIR / f"cepbrazil_postcode_conflicts_{SAVE_TAG}.csv",
    "cep_prefix_lookup_pkl": PROCESSED_DIR / f"cep_prefix_lookup_{SAVE_TAG}.pkl",
    "ufs_lookup_pkl": PROCESSED_DIR / f"ufs_regioes_lookup_{SAVE_TAG}.pkl",
    "ufs_lookup_csv": PROCESSED_DIR / f"ufs_regioes_lookup_{SAVE_TAG}.csv",
    "cep_usuarios_sem_match_pkl": PROCESSED_DIR / f"cep_usuarios_sem_match_{SAVE_TAG}.pkl",
    "ceps_cnes_sem_match_pkl": PROCESSED_DIR / f"ceps_cnes_sem_match_{SAVE_TAG}.pkl",
    "cep_usuarios_sem_match_csv": PROCESSED_DIR / f"cep_usuarios_sem_match_{SAVE_TAG}.csv",
    "ceps_cnes_sem_match_csv": PROCESSED_DIR / f"ceps_cnes_sem_match_{SAVE_TAG}.csv",
    "final_removed_rows_sample_csv": PROCESSED_DIR / f"final_removed_rows_sample_{SAVE_TAG}.csv",
}

# To avoid huge CSVs, save only a sample of the final data. The full final data is
# saved as pickle.
FINAL_SAMPLE_N = 10_000
REMOVED_SAMPLE_N = 10_000

# Strict final cleanup: after all enrichment, remove rows with any remaining
# missing/null-like/blank value in any remaining column.
STRICT_FINAL_DROP_ANY_MISSING = True

# Null/noise tokens found in real-world CSVs.
NULL_TOKENS = {
    "", " ", "na", "n/a", "nan", "none", "null", "nil", "missing",
    "sem informação", "sem informacao", "não informado", "nao informado",
    "ignorado", "indefinido", "-", "--", "---", ".", "?",
}

# Columns expected from the raw SISREG file after lowercasing.
RAW_REQUIRED_COLUMNS = [
    "cns_usuario", "dta_atend", "conf_atend", "cep_usuario", "cep_cnes_exe",
    "ibge_cnes_exe", "idade_usuario", "tempo_espera",
]

# These categorical columns are expected to be cleaned and preserved if present.
CATEGORICAL_COLUMNS = [
    "sexo_usuario", "raca_cor_usuario", "cbo_espec", "prioridade",
    "deslocamento", "tip_marcacao",
]

# Model-relevant columns expected at the end. Any row missing one of these is not
# usable for the experiments you are running.
FINAL_REQUIRED_COLUMNS = [
    "cns_usuario", "dta_atend", "ausencia",
    "idade_usuario", "tempo_espera",
    "cep_usuario", "cep_cnes_exe", "ibge_cnes_exe",
    "contagem_faltas_anteriores", "total_consultas_anteriores", "percentual_faltas_anteriores",
    "mesma_unidade_anterior", "dia_consulta", "mes_consulta", "ano_consulta",
    "lat_usuario", "lon_usuario", "cidade_usuario", "uf_usuario",
    "lat_cnes", "lon_cnes", "cidade_cnes", "uf_cnes",
    "cep_aprox_usuario", "cep_aprox_cnes",
    "mesma_cidade", "mesmo_estado", "mesma_regiao",
    "cod_uf_usuario", "cod_uf_cnes",
    "distance_km",
]

# Efficient raw loading dtypes. It keeps identifiers and CEPs as strings because
# numeric conversion destroys leading zeroes and may reject valid formatted CEPs.
SISREG_DTYPE_MAP = {
    "CNS_USUARIO": "string",
    "IBGE_USUARIO": "string",
    "CEP_USUARIO": "string",
    "IBGE_CNES_EXE": "string",
    "CEP_CNES_EXE": "string",
    "IDADE_USUARIO": "string",
    "TEMPO_ESPERA": "string",
    "SEXO_USUARIO": "string",
    "RACA_COR_USUARIO": "string",
    "CBO_ESPEC": "string",
    "PRIORIDADE": "string",
    "DESLOCAMENTO": "string",
    "TIP_MARCACAO": "string",
    "CONF_ATEND": "string",
}

# Static UF lookup avoids scraping public websites during preprocessing.
UF_LOOKUP_ROWS = [
    ("ac", 12, "norte"), ("al", 27, "nordeste"), ("ap", 16, "norte"),
    ("am", 13, "norte"), ("ba", 29, "nordeste"), ("ce", 23, "nordeste"),
    ("df", 53, "centro-oeste"), ("es", 32, "sudeste"), ("go", 52, "centro-oeste"),
    ("ma", 21, "nordeste"), ("mt", 51, "centro-oeste"), ("ms", 50, "centro-oeste"),
    ("mg", 31, "sudeste"), ("pa", 15, "norte"), ("pb", 25, "nordeste"),
    ("pr", 41, "sul"), ("pe", 26, "nordeste"), ("pi", 22, "nordeste"),
    ("rj", 33, "sudeste"), ("rn", 24, "nordeste"), ("rs", 43, "sul"),
    ("ro", 11, "norte"), ("rr", 14, "norte"), ("sc", 42, "sul"),
    ("sp", 35, "sudeste"), ("se", 28, "nordeste"), ("to", 17, "norte"),
]

CONFIG = {
    "SAVE_TAG": SAVE_TAG,
    "YEAR_FILTER": YEAR_FILTER,
    "PROJECT_ROOT": str(PROJECT_ROOT),
    "SISREG_FILE": str(SISREG_FILE),
    "CEP_FILE": str(CEP_FILE),
    "STRICT_FINAL_DROP_ANY_MISSING": STRICT_FINAL_DROP_ANY_MISSING,
    "FINAL_SAMPLE_N": FINAL_SAMPLE_N,
    "REMOVED_SAMPLE_N": REMOVED_SAMPLE_N,
    "OUTPUT_PATHS": {k: str(v) for k, v in OUTPUT_PATHS.items()},
}



In [63]:

# ==============================================================================
# LOGGING AND SMALL HELPERS
# ==============================================================================


In [64]:
# Print messages immediately with flush=True so long preprocessing steps show progress in real time.

def log(msg: str) -> None:
    print(msg, flush=True)



In [65]:
# Format integers with dot thousands separators, matching Brazilian-style readable logs.

def fmt_int(n) -> str:
    return f"{int(n):,}".replace(",", ".")


In [66]:
# Print a clear visual section header to make long notebook output easier to follow.

def print_section(title: str) -> None:
    log("\n" + "=" * 90)
    log(title)
    log("=" * 90)


In [67]:
# Standardize raw column names by lowercasing, removing accents, replacing symbols with underscores, and avoiding inconsistent CSV headers.

def normalize_column_name(col: str) -> str:
    col = str(col).strip().lower()
    col = unicodedata.normalize("NFKD", col).encode("ascii", "ignore").decode("ascii")
    col = re.sub(r"[^a-z0-9_]+", "_", col)
    col = re.sub(r"_+", "_", col).strip("_")
    return col



In [68]:
# Remove accents from text while preserving real missing values; useful for stable comparisons across Portuguese text.

def strip_accents(value):
    if pd.isna(value):
        return value
    return unicodedata.normalize("NFKD", str(value)).encode("ascii", "ignore").decode("ascii")



In [69]:
# Normalize one scalar text value by trimming, lowercasing, collapsing spaces, and converting null-like tokens into real missing values.

def normalize_text_value(value):
    """Normalize a scalar text value without converting real missing values to strings."""
    if pd.isna(value):
        return pd.NA
    text = str(value)
    text = text.replace("\u00a0", " ").replace("\ufeff", " ")
    text = unicodedata.normalize("NFKC", text)
    text = text.strip().lower()
    text = re.sub(r"\s+", " ", text)
    text_no_accents = strip_accents(text)
    if text in NULL_TOKENS or text_no_accents in NULL_TOKENS:
        return pd.NA
    return text



In [70]:
# Vectorized text normalization for full columns; optionally removes accents to make categorical values more consistent.

def normalize_text_series(series: pd.Series, strip_accents_output: bool = False) -> pd.Series:
    s = series.astype("string")
    s = s.str.replace("\u00a0", " ", regex=False).str.replace("\ufeff", " ", regex=False)
    s = s.str.normalize("NFKC")
    s = s.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)

    if strip_accents_output:
        s = s.map(lambda x: strip_accents(x) if not pd.isna(x) else pd.NA).astype("string")

    # Mark null-like tokens as real missing values.
    token_mask = s.isin(NULL_TOKENS)
    if strip_accents_output:
        token_mask = token_mask | s.map(lambda x: strip_accents(x) in NULL_TOKENS if not pd.isna(x) else False)
    s = s.mask(token_mask, pd.NA)
    return s.astype("string")



In [71]:
# Apply text normalization to every object/string/category column in a DataFrame.

def normalize_all_text_columns(df: pd.DataFrame, strip_accents_output: bool = False) -> pd.DataFrame:
    text_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    for col in text_cols:
        df[col] = normalize_text_series(df[col], strip_accents_output=strip_accents_output)
    return df



In [72]:
# Fail early if an expected column is missing, making data-source or schema problems easier to debug.

def ensure_required_columns(df: pd.DataFrame, required_cols: list[str], stage: str) -> None:
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns at stage '{stage}': {missing}")



In [73]:
# Save configuration or metadata as readable UTF-8 JSON for reproducibility.

def save_json(obj, path: Path) -> None:
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)



In [74]:
# Save unmatched/problematic values as both raw pickle data and a compact frequency table for inspection.

def save_series_counts(series: pd.Series, pkl_path: Path, csv_path: Path, name: str) -> None:
    series = series.dropna().astype("string")
    series.to_pickle(pkl_path)
    counts = series.value_counts(dropna=False).rename_axis(name).reset_index(name="count")
    counts.to_csv(csv_path, index=False)



In [75]:

# ==============================================================================
# DATA QUALITY AUDIT HELPERS
# ==============================================================================


In [76]:
# Build a column-level quality report with missing values, blank strings, null-like strings, whitespace problems, infinite values, and unique counts.

def build_quality_audit(df: pd.DataFrame, stage: str) -> pd.DataFrame:
    """Create a column-level audit for missing, null-like and whitespace noise."""
    total_rows = len(df)
    rows = []

    for col in df.columns:
        s = df[col]
        dtype = str(s.dtype)
        missing_count = int(s.isna().sum())
        blank_count = 0
        null_like_string_count = 0
        leading_trailing_ws_count = 0
        internal_multi_ws_count = 0
        inf_count = 0

        if pd.api.types.is_numeric_dtype(s):
            arr = pd.to_numeric(s, errors="coerce")
            inf_count = int(np.isinf(arr.to_numpy(dtype="float64", na_value=np.nan)).sum())

        if pd.api.types.is_object_dtype(s) or pd.api.types.is_string_dtype(s) or isinstance(s.dtype, pd.CategoricalDtype):
            st = s.astype("string")
            raw_non_missing = st.dropna()
            stripped = raw_non_missing.str.strip()
            lowered = stripped.str.lower()
            lowered_no_accents = lowered.map(lambda x: strip_accents(x) if not pd.isna(x) else x)

            blank_count = int((stripped == "").sum())
            null_like_string_count = int((lowered.isin(NULL_TOKENS) | lowered_no_accents.isin(NULL_TOKENS)).sum())
            leading_trailing_ws_count = int((raw_non_missing != stripped).sum())
            internal_multi_ws_count = int(raw_non_missing.str.contains(r"\s{2,}", regex=True, na=False).sum())

        rows.append({
            "stage": stage,
            "column": col,
            "dtype": dtype,
            "rows": total_rows,
            "missing_count": missing_count,
            "missing_pct": (missing_count / total_rows * 100) if total_rows else 0.0,
            "blank_string_count": blank_count,
            "null_like_string_count": null_like_string_count,
            "leading_trailing_whitespace_count": leading_trailing_ws_count,
            "internal_multiple_whitespace_count": internal_multi_ws_count,
            "infinite_count": inf_count,
            "unique_count_including_na": int(s.nunique(dropna=False)),
        })

    return pd.DataFrame(rows)



In [77]:
# Run the quality audit, save it to CSV, and log the columns that still contain data-quality issues.

def audit_and_save(df: pd.DataFrame, stage: str, path: Path) -> pd.DataFrame:
    audit = build_quality_audit(df, stage)
    audit.to_csv(path, index=False)
    log(f"Saved quality audit [{stage}]: {path}")

    problematic = audit[
        (audit["missing_count"] > 0)
        | (audit["blank_string_count"] > 0)
        | (audit["null_like_string_count"] > 0)
        | (audit["leading_trailing_whitespace_count"] > 0)
        | (audit["internal_multiple_whitespace_count"] > 0)
        | (audit["infinite_count"] > 0)
    ]
    if problematic.empty:
        log(f"✅ Audit [{stage}]: no missing/null-like/whitespace/infinite problems detected.")
    else:
        log(f"⚠️ Audit [{stage}]: found {len(problematic)} columns with potential quality issues.")
        log(str(problematic[["column", "missing_count", "blank_string_count", "null_like_string_count", "leading_trailing_whitespace_count", "infinite_count"]].head(30)))
    return audit



In [78]:
# Build the final strict row-removal mask and a reason summary, covering missing values, invalid targets, invalid flags, impossible domains, and coordinate bounds.

def make_row_drop_report(df: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:
    """Build final row-removal mask and reason summary."""
    reasons = {}
    total = len(df)

    # Missing in required columns.
    existing_required = [c for c in FINAL_REQUIRED_COLUMNS if c in df.columns]
    missing_required = df[existing_required].isna().any(axis=1)
    reasons["missing_in_final_required_columns"] = missing_required

    # Missing anywhere, if strict cleanup is enabled.
    if STRICT_FINAL_DROP_ANY_MISSING:
        reasons["missing_any_column_strict"] = df.isna().any(axis=1)

    # Infinite numeric values.
    numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
    if numeric_cols:
        numeric_arr = df[numeric_cols].to_numpy(dtype="float64", na_value=np.nan)
        reasons["infinite_numeric_value"] = pd.Series(np.isinf(numeric_arr).any(axis=1), index=df.index)
    else:
        reasons["infinite_numeric_value"] = pd.Series(False, index=df.index)

    # Invalid target.
    if TARGET_COL in df.columns:
        reasons["invalid_target"] = ~df[TARGET_COL].isin([0, 1])

    # Invalid binary flags.
    for col in ["mesma_unidade_anterior", "cep_aprox_usuario", "cep_aprox_cnes", "mesma_cidade", "mesmo_estado", "mesma_regiao"]:
        if col in df.columns:
            reasons[f"invalid_binary_{col}"] = ~df[col].isin([0, 1])

    # Domain sanity checks.
    if "idade_usuario" in df.columns:
        reasons["invalid_idade_usuario"] = (df["idade_usuario"] < 0) | (df["idade_usuario"] > 130)
    if "tempo_espera" in df.columns:
        reasons["invalid_tempo_espera"] = df["tempo_espera"] < 0
    if "percentual_faltas_anteriores" in df.columns:
        reasons["invalid_percentual_faltas_anteriores"] = (df["percentual_faltas_anteriores"] < 0) | (df["percentual_faltas_anteriores"] > 1)
    if "distance_km" in df.columns:
        reasons["invalid_distance_km"] = (df["distance_km"] < 0) | (~np.isfinite(df["distance_km"].astype("float64")))

    # Brazil coordinate bounds, inclusive and intentionally broad.
    coord_bounds = {
        "lat_usuario": (-35.0, 6.5),
        "lat_cnes": (-35.0, 6.5),
        "lon_usuario": (-75.0, -30.0),
        "lon_cnes": (-75.0, -30.0),
    }
    for col, (lo, hi) in coord_bounds.items():
        if col in df.columns:
            reasons[f"invalid_coordinate_{col}"] = (df[col] < lo) | (df[col] > hi)

    reason_df = pd.DataFrame({k: v.astype(bool) for k, v in reasons.items()}, index=df.index)
    drop_mask = reason_df.any(axis=1)

    summary_rows = []
    for reason in reason_df.columns:
        count = int(reason_df[reason].sum())
        summary_rows.append({
            "reason": reason,
            "rows_flagged": count,
            "pct_of_current_rows": (count / total * 100) if total else 0.0,
        })
    summary_rows.append({
        "reason": "ANY_REASON_TOTAL_UNIQUE_ROWS",
        "rows_flagged": int(drop_mask.sum()),
        "pct_of_current_rows": (drop_mask.sum() / total * 100) if total else 0.0,
    })

    return drop_mask, pd.DataFrame(summary_rows)



In [79]:

# ==============================================================================
# DOMAIN CLEANING HELPERS
# ==============================================================================


In [80]:
# Normalize CEP values into valid 8-digit strings, preserving leading zeros and rejecting malformed postcodes.

def clean_cep_to_string(series: pd.Series) -> pd.Series:
    """Return normalized 8-digit CEP strings, preserving leading zeros."""
    s = series.astype("string")
    s = s.str.replace(r"\.0$", "", regex=True)
    s = s.str.replace(r"\D+", "", regex=True)
    s = s.mask(s.str.len() == 0, pd.NA)

    # If someone stored a valid CEP with missing leading zero, restore it.
    s = s.mask(s.str.len() == 7, "0" + s)

    # Keep only exact 8-digit CEPs after normalization.
    valid = s.str.fullmatch(r"\d{8}", na=False)
    return s.where(valid, pd.NA).astype("string")



In [81]:
# Remove non-digit characters from identifier-like fields while keeping them as strings to avoid losing leading zeros.

def clean_digits_to_string(series: pd.Series, min_len: int | None = None, max_len: int | None = None) -> pd.Series:
    s = series.astype("string").str.replace(r"\.0$", "", regex=True).str.replace(r"\D+", "", regex=True)
    s = s.mask(s.str.len() == 0, pd.NA)
    if min_len is not None:
        s = s.where(s.str.len() >= min_len, pd.NA)
    if max_len is not None:
        s = s.where(s.str.len() <= max_len, pd.NA)
    return s.astype("string")



In [82]:
# Pseudonymize patient identifiers with SHA-256 while preserving uniqueness checks and avoiding exposure of raw CNS values.

def hash_identifier(series: pd.Series) -> pd.Series:
    def _hash_one(x):
        if pd.isna(x):
            return pd.NA
        return hashlib.sha256(str(x).encode("utf-8")).hexdigest()
    return series.map(_hash_one).astype("string")



In [83]:
# Print and optionally save the target-class distribution, useful for checking the absence/no-show imbalance.

def print_class_distribution(df: pd.DataFrame, col: str, path: Path | None = None) -> pd.DataFrame:
    counts = df[col].value_counts(dropna=False).sort_index()
    pct = df[col].value_counts(normalize=True, dropna=False).sort_index() * 100
    out = pd.DataFrame({"class": counts.index.astype(str), "count": counts.values, "percentage": pct.values})
    log("\nClass distribution:")
    log(str(out))
    if path is not None:
        out.to_csv(path, index=False)
        log(f"Saved class distribution: {path}")
    return out



In [84]:
# Build a static Brazilian UF lookup table with state abbreviations, state codes, and macro-regions, avoiding external web dependencies during preprocessing.

def build_uf_lookup() -> pd.DataFrame:
    df_uf = pd.DataFrame(UF_LOOKUP_ROWS, columns=["sigla", "codigo_uf", "regiao"])
    df_uf["sigla"] = df_uf["sigla"].astype("string")
    df_uf["codigo_uf"] = df_uf["codigo_uf"].astype("int16")
    df_uf["regiao"] = df_uf["regiao"].astype("string")
    return df_uf



In [85]:
# Return the modal value of a series, or missing if no value exists; used when summarizing location prefixes.

def mode_or_na(series: pd.Series):
    s = series.dropna()
    if s.empty:
        return pd.NA
    return s.mode().iloc[0]


In [86]:
# Build an in-memory CEP-prefix lookup using median coordinates and modal city/UF; useful as a deterministic prefix-recovery strategy, although the later memory-safe version is preferred for large CEP files.

def build_prefix_lookup(cep_coords: pd.DataFrame, min_prefix_len: int = 5) -> dict[str, dict]:
    """Build deterministic prefix lookup using grouped median coords and modal city/UF."""
    lookup = {}
    base = cep_coords[["postcode", "lat", "lon", "city", "region"]].copy()
    base["postcode"] = base["postcode"].astype("string")

    for length in range(8, min_prefix_len - 1, -1):
        tmp = base.copy()
        tmp["prefix"] = tmp["postcode"].str[:length]
        grouped = tmp.groupby("prefix", observed=True).agg(
            lat=("lat", "median"),
            lon=("lon", "median"),
            city=("city", mode_or_na),
            region=("region", mode_or_na),
            n_postcodes=("postcode", "nunique"),
        ).reset_index()

        for row in grouped.itertuples(index=False):
            prefix = row.prefix
            if prefix not in lookup:
                lookup[prefix] = {
                    "lat": float(row.lat),
                    "lon": float(row.lon),
                    "city": row.city,
                    "region": row.region,
                    "prefix_len": int(length),
                    "n_postcodes_in_prefix": int(row.n_postcodes),
                }
    return lookup



In [87]:
# Recover approximate location data from the longest available CEP prefix when exact CEP matching is unavailable.

def recover_location_by_prefix(cep_series: pd.Series, prefix_lookup: dict[str, dict]) -> pd.DataFrame:
    rows = []
    for idx, cep in cep_series.items():
        if pd.isna(cep):
            continue
        cep_str = str(cep)
        match = None
        for length in range(8, 4, -1):
            prefix = cep_str[:length]
            if prefix in prefix_lookup:
                match = prefix_lookup[prefix]
                break
        if match is not None:
            rows.append({
                "index": idx,
                "lat": match["lat"],
                "lon": match["lon"],
                "city": match["city"],
                "region": match["region"],
                "prefix_len": match["prefix_len"],
                "n_postcodes_in_prefix": match["n_postcodes_in_prefix"],
            })
    if not rows:
        return pd.DataFrame(columns=["lat", "lon", "city", "region", "prefix_len", "n_postcodes_in_prefix"])
    return pd.DataFrame(rows).set_index("index")



In [88]:
# Compute vectorized Haversine distance in kilometers between user and health-unit coordinates.

def calculate_distance_km(lat1, lon1, lat2, lon2) -> np.ndarray:
    """Vectorized Haversine distance in kilometers."""
    r = 6371.0
    lat1_rad, lon1_rad = np.radians(lat1.astype("float64")), np.radians(lon1.astype("float64"))
    lat2_rad, lon2_rad = np.radians(lat2.astype("float64")), np.radians(lon2.astype("float64"))
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon / 2) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return r * c



In [89]:
# Convert final columns to memory-efficient dtypes such as category, nullable integers, string, and float32 after cleaning is complete.

def convert_final_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """Apply memory-efficient dtypes after all cleaning is complete."""
    string_cols = [
        "cns_usuario", "ibge_usuario", "cep_usuario", "ibge_cnes_exe", "cep_cnes_exe",
        "cidade_usuario", "uf_usuario", "cidade_cnes", "uf_cnes", "regiao_usuario", "regiao_cnes",
    ]
    cat_cols = [c for c in CATEGORICAL_COLUMNS if c in df.columns]
    int8_cols = [
        "ausencia", "mesma_unidade_anterior", "cep_aprox_usuario", "cep_aprox_cnes",
        "cep_prefix_len_usuario", "cep_prefix_len_cnes",
        "mesma_cidade", "mesmo_estado", "mesma_regiao", "dia_consulta", "mes_consulta",
    ]
    int16_cols = ["idade_usuario", "tempo_espera", "ano_consulta", "cod_uf_usuario", "cod_uf_cnes"]
    int32_cols = ["contagem_faltas_anteriores", "total_consultas_anteriores"]
    float32_cols = [
        "percentual_faltas_anteriores", "distance_km", "lat_usuario", "lon_usuario", "lat_cnes", "lon_cnes",
    ]

    for col in string_cols:
        if col in df.columns:
            df[col] = df[col].astype("string")
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype("category")
    for col in int8_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int8")
    for col in int16_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int16")
    for col in int32_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int32")
    for col in float32_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

    df["dta_atend"] = pd.to_datetime(df["dta_atend"], errors="coerce")
    return df



In [90]:

# ==============================================================================
# LOAD AND CLEAN RAW SISREG
# ==============================================================================


In [91]:
# Load the raw SISREG CSV with safe string dtypes, normalize column names, validate required columns, and save the initial quality audit.

def load_sisreg() -> pd.DataFrame:
    print_section("LOAD RAW SISREG")
    if not SISREG_FILE.exists():
        raise FileNotFoundError(f"SISREG file not found: {SISREG_FILE}")

    log(f"Loading: {SISREG_FILE}")
    df = pd.read_csv(
        SISREG_FILE,
        on_bad_lines="skip",
        engine="python",
        dtype=SISREG_DTYPE_MAP,
    )
    log(f"Loaded raw shape: {df.shape}")
    df.columns = [normalize_column_name(c) for c in df.columns]
    ensure_required_columns(df, RAW_REQUIRED_COLUMNS, "raw load")
    audit_and_save(df, "initial_raw", OUTPUT_PATHS["quality_initial_csv"])
    return df



In [92]:
# Perform core SISREG cleaning: normalize text, parse dates, map attendance into the binary target, drop invalid keys/domains, clean CEP/IBGE/CNS fields, hash CNS identifiers, and audit the result.

def clean_basic_sisreg(df: pd.DataFrame) -> pd.DataFrame:
    print_section("BASIC SISREG CLEANING")
    start_rows = len(df)

    # Normalize all text early. strip_accents_output=True makes comparisons more stable.
    df = normalize_all_text_columns(df, strip_accents_output=True)

    # Dates.
    df["dta_atend"] = pd.to_datetime(df["dta_atend"], errors="coerce")

    # Target: validate explicitly. Never silently convert unknown classes to 0.
    df["conf_atend"] = normalize_text_series(df["conf_atend"], strip_accents_output=True)
    target_map = {
        "nao": 1, "n": 1, "no": 1,      # no-show/absence in your original coding
        "sim": 0, "s": 0, "yes": 0,     # attended/present
    }
    valid_target_mask = df["conf_atend"].isin(target_map.keys())
    invalid_target_count = int((~valid_target_mask).sum())
    if invalid_target_count:
        log(f"Dropping {fmt_int(invalid_target_count)} rows with invalid/unknown conf_atend values.")
    df = df.loc[valid_target_mask].copy()
    df[TARGET_COL] = df["conf_atend"].map(target_map).astype("Int8")
    df.drop(columns=["conf_atend"], inplace=True)

    # Critical identifiers and date.
    required_non_empty = ["cns_usuario", "dta_atend"]
    before = len(df)
    df = df.dropna(subset=required_non_empty).copy()
    log(f"Dropped {fmt_int(before - len(df))} rows missing cns_usuario or dta_atend.")

    # Clean identifiers/merge keys.
    df["cns_usuario"] = clean_digits_to_string(df["cns_usuario"], min_len=1)
    df["cep_usuario"] = clean_cep_to_string(df["cep_usuario"])
    df["cep_cnes_exe"] = clean_cep_to_string(df["cep_cnes_exe"])
    df["ibge_usuario"] = clean_digits_to_string(df["ibge_usuario"], min_len=1) if "ibge_usuario" in df.columns else pd.NA
    df["ibge_cnes_exe"] = clean_digits_to_string(df["ibge_cnes_exe"], min_len=1)

    before = len(df)
    df = df.dropna(subset=["cns_usuario", "cep_usuario", "cep_cnes_exe", "ibge_cnes_exe"]).copy()
    log(f"Dropped {fmt_int(before - len(df))} rows with invalid CNS/CEP/IBGE merge keys.")

    # Numeric fields.
    df["idade_usuario"] = pd.to_numeric(df["idade_usuario"], errors="coerce")
    df["tempo_espera"] = pd.to_numeric(df["tempo_espera"], errors="coerce")

    before = len(df)
    domain_mask = (
        df["idade_usuario"].between(0, 130, inclusive="both")
        & (df["tempo_espera"] >= 0)
        & df["idade_usuario"].notna()
        & df["tempo_espera"].notna()
    )
    df = df.loc[domain_mask].copy()
    log(f"Dropped {fmt_int(before - len(df))} rows with invalid idade_usuario/tempo_espera.")

    # Pseudonymize after cleaning so null-like strings do not get hashed as real IDs.
    before_unique = df["cns_usuario"].nunique(dropna=True)
    df["cns_usuario"] = hash_identifier(df["cns_usuario"])
    after_unique = df["cns_usuario"].nunique(dropna=True)
    if before_unique != after_unique:
        raise RuntimeError(f"Hash unique-count mismatch: before={before_unique}, after={after_unique}")
    log(f"Hashed CNS identifiers. Unique patients preserved: {fmt_int(after_unique)}")

    # Categorical columns normalized as strings for now; categories are assigned at final dtype conversion.
    for col in CATEGORICAL_COLUMNS:
        if col in df.columns:
            df[col] = normalize_text_series(df[col], strip_accents_output=True)

    audit_and_save(df, "after_basic_cleaning", OUTPUT_PATHS["quality_after_basic_csv"])
    log(f"Basic cleaning complete. Rows: {fmt_int(start_rows)} -> {fmt_int(len(df))}")
    print_class_distribution(df, TARGET_COL, OUTPUT_PATHS["class_distribution_csv"])
    return df



In [93]:

# ==============================================================================
# HISTORICAL FEATURES
# ==============================================================================


In [94]:
# Create patient-history features using only previous appointments, avoiding leakage from the current row's target.

def add_historical_features(df: pd.DataFrame) -> pd.DataFrame:
    print_section("HISTORICAL FEATURES")
    df = df.copy()
    df["__original_order"] = np.arange(len(df), dtype=np.int64)
    df = df.sort_values(["cns_usuario", "dta_atend", "__original_order"], kind="mergesort").copy()

    # Prior absences: cumsum includes current row, so subtract current target to avoid leakage.
    cumulative_absences = df.groupby("cns_usuario", observed=True)[TARGET_COL].cumsum()
    df["contagem_faltas_anteriores"] = (cumulative_absences - df[TARGET_COL]).astype("int32")
    df["total_consultas_anteriores"] = df.groupby("cns_usuario", observed=True).cumcount().astype("int32")
    df["percentual_faltas_anteriores"] = np.where(
        df["total_consultas_anteriores"] > 0,
        df["contagem_faltas_anteriores"] / df["total_consultas_anteriores"],
        0.0,
    ).astype("float32")

    prev_unit = df.groupby("cns_usuario", observed=True)[["ibge_cnes_exe", "cep_cnes_exe"]].shift(1)
    same_unit = (df["ibge_cnes_exe"] == prev_unit["ibge_cnes_exe"]) & (df["cep_cnes_exe"] == prev_unit["cep_cnes_exe"])
    df["mesma_unidade_anterior"] = same_unit.fillna(False).astype("int8")

    df.drop(columns=["__original_order"], inplace=True)
    log("Historical features created without using the current row's target.")
    return df



In [95]:
# Keep only the configured analysis year and add day, month, and year appointment features.

def filter_year_and_add_date_parts(df: pd.DataFrame, year: int) -> pd.DataFrame:
    print_section(f"FILTER TO YEAR {year}")
    start = pd.Timestamp(f"{year}-01-01")
    end = pd.Timestamp(f"{year}-12-31")
    before = len(df)
    df_year = df.loc[df["dta_atend"].between(start, end, inclusive="both")].copy()
    log(f"Filtered rows: {fmt_int(before)} -> {fmt_int(len(df_year))}")

    df_year["dia_consulta"] = df_year["dta_atend"].dt.day.astype("int8")
    df_year["mes_consulta"] = df_year["dta_atend"].dt.month.astype("int8")
    df_year["ano_consulta"] = df_year["dta_atend"].dt.year.astype("int16")
    return df_year



In [96]:

# ============================================================================== 
# CEP LOOKUP AND LOCATION ENRICHMENT - MEMORY-SAFE
# ==============================================================================


In [97]:
# Clean one chunk of cepbrazil.csv by normalizing postcode/city/UF and validating Brazilian coordinate bounds without loading the full file into memory.

def clean_cep_chunk(raw_chunk: pd.DataFrame) -> pd.DataFrame:
    """Clean one cepbrazil.csv chunk without materializing the full lookup table."""
    chunk = raw_chunk.copy()
    chunk.columns = [normalize_column_name(c) for c in chunk.columns]

    # Keep the operations column-specific. normalize_all_text_columns on the full
    # CEP file was the major RAM spike before.
    chunk["postcode"] = clean_cep_to_string(chunk["postcode"])
    chunk["city"] = normalize_text_series(chunk["city"], strip_accents_output=True)
    chunk["region"] = normalize_text_series(chunk["region"], strip_accents_output=True)
    chunk["lat"] = pd.to_numeric(chunk["lat"], errors="coerce")
    chunk["lon"] = pd.to_numeric(chunk["lon"], errors="coerce")

    valid = (
        chunk["postcode"].notna()
        & chunk["lat"].between(-35.0, 6.5, inclusive="both")
        & chunk["lon"].between(-75.0, -30.0, inclusive="both")
        & chunk["city"].notna()
        & chunk["region"].notna()
    )
    chunk = chunk.loc[valid, ["postcode", "lat", "lon", "city", "region"]]

    # Compact dtypes before returning the chunk.
    chunk["lat"] = chunk["lat"].astype("float32")
    chunk["lon"] = chunk["lon"].astype("float32")
    chunk["postcode"] = chunk["postcode"].astype("string")
    chunk["city"] = chunk["city"].astype("string")
    chunk["region"] = chunk["region"].astype("string")
    return chunk



In [98]:
# Stream valid cleaned CEP chunks from cepbrazil.csv, keeping the lookup process memory-safe.

def iter_clean_cep_chunks(chunksize: int = CEP_CHUNK_SIZE):
    """Yield valid, compact CEP chunks from cepbrazil.csv."""
    cols = ["LON", "LAT", "CITY", "REGION", "POSTCODE"]
    for raw_chunk in pd.read_csv(
        CEP_FILE,
        usecols=cols,
        dtype="string",
        chunksize=chunksize,
        low_memory=True,
    ):
        yield clean_cep_chunk(raw_chunk)



In [99]:
# Collect only the unique user and CNES CEPs that actually appear in SISREG, reducing the CEP lookup workload.

def build_needed_cep_set(df: pd.DataFrame) -> set[str]:
    """Return unique CEPs that are actually used in the SISREG data."""
    user_ceps = df["cep_usuario"].dropna().astype("string")
    cnes_ceps = df["cep_cnes_exe"].dropna().astype("string")
    needed = pd.unique(pd.concat([user_ceps, cnes_ceps], ignore_index=True))
    needed_set = {str(x) for x in needed if not pd.isna(x)}
    log(f"Unique CEPs needed by SISREG after year filter: {fmt_int(len(needed_set))}")
    return needed_set



In [100]:
# Accumulate prefix-level coordinate sums and city/UF counts across chunks without storing every matching CEP row.

def update_prefix_stats(prefix_stats: dict, subset: pd.DataFrame, prefix_col: str, prefix_len: int) -> None:
    """Aggregate chunk-level prefix candidates without storing every matching row."""
    if subset.empty:
        return

    # Mean coordinates are used for memory safety. In previous versions we used median on the full
    # lookup, which required holding the entire CEP table in memory.
    coord = subset.groupby(prefix_col, observed=True).agg(
        lat_sum=("lat", "sum"),
        lon_sum=("lon", "sum"),
        n_rows=("postcode", "size"),
        n_postcodes=("postcode", "nunique"),
    )

    for prefix, row in coord.iterrows():
        d = prefix_stats.setdefault(
            str(prefix),
            {
                "prefix_len": int(prefix_len),
                "lat_sum": 0.0,
                "lon_sum": 0.0,
                "n_rows": 0,
                "n_postcodes": 0,
                "city_counts": Counter(),
                "region_counts": Counter(),
            },
        )
        d["lat_sum"] += float(row["lat_sum"])
        d["lon_sum"] += float(row["lon_sum"])
        d["n_rows"] += int(row["n_rows"])
        d["n_postcodes"] += int(row["n_postcodes"])

    city_counts = subset.groupby([prefix_col, "city"], observed=True).size()
    for (prefix, city), count in city_counts.items():
        prefix_stats[str(prefix)]["city_counts"][str(city)] += int(count)

    region_counts = subset.groupby([prefix_col, "region"], observed=True).size()
    for (prefix, region), count in region_counts.items():
        prefix_stats[str(prefix)]["region_counts"][str(region)] += int(count)



In [101]:
# Convert accumulated prefix statistics into a lookup dictionary with mean coordinates, modal city/UF, prefix length, and support counts.

def finalize_prefix_lookup(prefix_stats: dict) -> dict[str, dict]:
    lookup = {}
    for prefix, stats in prefix_stats.items():
        n = max(int(stats["n_rows"]), 1)
        city = stats["city_counts"].most_common(1)[0][0] if stats["city_counts"] else pd.NA
        region = stats["region_counts"].most_common(1)[0][0] if stats["region_counts"] else pd.NA
        lookup[prefix] = {
            "lat": float(stats["lat_sum"] / n),
            "lon": float(stats["lon_sum"] / n),
            "city": city,
            "region": region,
            "prefix_len": int(stats["prefix_len"]),
            "n_rows_in_prefix": int(stats["n_rows"]),
            "n_postcodes_in_prefix": int(stats["n_postcodes"]),
        }
    return lookup



In [102]:
# Build exact CEP matches and targeted prefix-recovery matches in two streaming passes, saving conflicts and avoiding a full in-memory CEP merge.

def load_and_clean_cep_resources(needed_ceps: set[str]) -> tuple[pd.DataFrame, dict[str, dict]]:
    """Build exact CEP lookup and targeted prefix lookup in memory-safe chunks.

    It performs two streaming passes:
      1) exact lookup only for CEPs present in SISREG;
      2) prefix lookup only for CEPs still unmatched after exact lookup.
    """
    print_section("LOAD AND CLEAN CEP LOOKUP - MEMORY-SAFE")
    if not CEP_FILE.exists():
        raise FileNotFoundError(f"CEP lookup file not found: {CEP_FILE}")

    log(f"Loading CEP lookup in chunks of {fmt_int(CEP_CHUNK_SIZE)} rows.")
    log(f"Exact lookup target CEPs: {fmt_int(len(needed_ceps))}")

    exact_rows = {}
    conflict_rows = []
    total_conflict_rows_seen = 0
    total_valid_rows = 0
    total_exact_rows_seen = 0

    for i, chunk in enumerate(iter_clean_cep_chunks(), start=1):
        total_valid_rows += len(chunk)
        if i % 10 == 0:
            log(f"  Pass 1 chunk {i}: valid rows seen={fmt_int(total_valid_rows)} | exact CEPs found={fmt_int(len(exact_rows))}")

        sub = chunk.loc[chunk["postcode"].isin(needed_ceps)]
        total_exact_rows_seen += len(sub)
        if sub.empty:
            continue

        for row in sub.itertuples(index=False):
            postcode = str(row.postcode)
            current = (float(row.lat), float(row.lon), str(row.city), str(row.region))
            previous = exact_rows.get(postcode)
            if previous is None:
                exact_rows[postcode] = current
            elif previous != current:
                total_conflict_rows_seen += 1
                if len(conflict_rows) < MAX_CEP_CONFLICT_REPORT_ROWS:
                    conflict_rows.append({
                        "postcode": postcode,
                        "kept_lat": previous[0],
                        "kept_lon": previous[1],
                        "kept_city": previous[2],
                        "kept_region": previous[3],
                        "other_lat": current[0],
                        "other_lon": current[1],
                        "other_city": current[2],
                        "other_region": current[3],
                    })

        del sub, chunk
        gc.collect()

    exact_df = pd.DataFrame(
        [
            {"postcode": p, "lat": v[0], "lon": v[1], "city": v[2], "region": v[3]}
            for p, v in exact_rows.items()
        ]
    )
    if exact_df.empty:
        raise RuntimeError("No exact CEP matches found. Check cepbrazil.csv and CEP cleaning rules.")

    exact_df["postcode"] = exact_df["postcode"].astype("string")
    exact_df["lat"] = exact_df["lat"].astype("float32")
    exact_df["lon"] = exact_df["lon"].astype("float32")
    exact_df["city"] = exact_df["city"].astype("string")
    exact_df["region"] = exact_df["region"].astype("string")
    exact_df.to_pickle(OUTPUT_PATHS["cep_lookup_clean_pkl"])
    log(f"Saved relevant clean CEP lookup: {OUTPUT_PATHS['cep_lookup_clean_pkl']} | rows={fmt_int(len(exact_df))}")
    log(f"Exact CEP rows seen before dedupe: {fmt_int(total_exact_rows_seen)}")

    conflicts_df = pd.DataFrame(conflict_rows).drop_duplicates() if conflict_rows else pd.DataFrame(
        columns=["postcode", "kept_lat", "kept_lon", "kept_city", "kept_region", "other_lat", "other_lon", "other_city", "other_region"]
    )
    conflicts_df.to_csv(OUTPUT_PATHS["cep_conflicts_csv"], index=False)
    log(
        f"Saved relevant CEP conflict report: {OUTPUT_PATHS['cep_conflicts_csv']} | "
        f"sampled_conflicts={fmt_int(len(conflicts_df))} | total_conflicting_rows_seen={fmt_int(total_conflict_rows_seen)}"
    )

    # Targeted prefix recovery only for CEPs that were needed but not exact-matched.
    exact_found = set(exact_rows.keys())
    missing_exact_ceps = sorted(needed_ceps - exact_found)
    log(f"Needed CEPs without exact lookup match: {fmt_int(len(missing_exact_ceps))}")

    target_prefixes_by_len = defaultdict(set)
    for cep in missing_exact_ceps:
        for length in range(7, CEP_PREFIX_MIN_LEN - 1, -1):
            if len(cep) >= length:
                target_prefixes_by_len[length].add(cep[:length])

    total_target_prefixes = sum(len(v) for v in target_prefixes_by_len.values())
    log(f"Targeted prefix candidates to search: {fmt_int(total_target_prefixes)}")

    prefix_stats = {}
    if total_target_prefixes > 0:
        for i, chunk in enumerate(iter_clean_cep_chunks(), start=1):
            if i % 10 == 0:
                log(f"  Pass 2 chunk {i}: prefix entries currently found={fmt_int(len(prefix_stats))}")

            for length, target_prefixes in target_prefixes_by_len.items():
                if not target_prefixes:
                    continue
                prefix_col = f"__prefix_{length}"
                prefixes = chunk["postcode"].str[:length]
                mask = prefixes.isin(target_prefixes)
                if mask.any():
                    sub = chunk.loc[mask, ["postcode", "lat", "lon", "city", "region"]].copy()
                    sub[prefix_col] = prefixes.loc[mask].astype("string").values
                    update_prefix_stats(prefix_stats, sub, prefix_col, length)
                    del sub

            del chunk
            gc.collect()

    prefix_lookup = finalize_prefix_lookup(prefix_stats)
    pd.to_pickle(prefix_lookup, OUTPUT_PATHS["cep_prefix_lookup_pkl"])
    log(f"Saved targeted CEP prefix lookup: {OUTPUT_PATHS['cep_prefix_lookup_pkl']} | prefixes={fmt_int(len(prefix_lookup))}")

    return exact_df, prefix_lookup



In [103]:
# Map unmatched CEPs to recovered locations using the longest available prefix, prioritizing more specific prefixes first.

def build_prefix_recovery_maps(unique_ceps: pd.Series, prefix_lookup: dict[str, dict]) -> dict[str, dict]:
    """Return CEP -> recovered location dictionary using longest available prefix."""
    recovery = {}
    for cep in unique_ceps.dropna().astype("string").unique():
        cep_str = str(cep)
        for length in range(7, CEP_PREFIX_MIN_LEN - 1, -1):
            prefix = cep_str[:length]
            match = prefix_lookup.get(prefix)
            if match is not None:
                recovery[cep_str] = match
                break
    return recovery



In [104]:
# Fill missing user or CNES location fields from prefix recovery, mark them as approximate, store the prefix length used, and return the number recovered.

def apply_prefix_recovery(
    df: pd.DataFrame,
    missing_mask: pd.Series,
    cep_col: str,
    lat_col: str,
    lon_col: str,
    city_col: str,
    uf_col: str,
    approx_col: str,
    prefix_len_col: str,
    prefix_lookup: dict[str, dict],
    label: str,
) -> int:
    if not missing_mask.any() or not prefix_lookup:
        return 0

    unique_missing = df.loc[missing_mask, cep_col].dropna().astype("string")
    recovery_maps = build_prefix_recovery_maps(unique_missing, prefix_lookup)
    if not recovery_maps:
        return 0

    lat_map = {k: v["lat"] for k, v in recovery_maps.items()}
    lon_map = {k: v["lon"] for k, v in recovery_maps.items()}
    city_map = {k: v["city"] for k, v in recovery_maps.items()}
    uf_map = {k: v["region"] for k, v in recovery_maps.items()}
    prefix_map = {k: v["prefix_len"] for k, v in recovery_maps.items()}

    target_ceps = df.loc[missing_mask, cep_col].astype("string")
    recovered_lat = target_ceps.map(lat_map)
    recovered_mask = recovered_lat.notna()
    if not recovered_mask.any():
        return 0

    idx = recovered_mask[recovered_mask].index
    df.loc[idx, lat_col] = recovered_lat.loc[idx].astype("float32").values
    df.loc[idx, lon_col] = target_ceps.loc[idx].map(lon_map).astype("float32").values
    df.loc[idx, city_col] = target_ceps.loc[idx].map(city_map).astype("string").values
    df.loc[idx, uf_col] = target_ceps.loc[idx].map(uf_map).astype("string").values
    df.loc[idx, approx_col] = 1
    df.loc[idx, prefix_len_col] = target_ceps.loc[idx].map(prefix_map).astype("Int8").values

    log(f"Recovered {label} locations by CEP prefix: {fmt_int(len(idx))}")
    return int(len(idx))



In [105]:
# Add user and CNES coordinates/city/UF via exact CEP maps first, then prefix recovery, save unmatched CEP reports, and drop rows still missing required location data.

def enrich_locations(df: pd.DataFrame, cep_exact: pd.DataFrame, prefix_lookup: dict[str, dict]) -> pd.DataFrame:
    print_section("LOCATION ENRICHMENT - MEMORY-SAFE")

    # Avoid two huge DataFrame merges. Series.map creates only the destination
    # columns and is much lighter for 8M+ rows.
    cep_indexed = cep_exact.drop_duplicates("postcode").set_index("postcode")

    lat_map = cep_indexed["lat"]
    lon_map = cep_indexed["lon"]
    city_map = cep_indexed["city"]
    region_map = cep_indexed["region"]

    df["lat_usuario"] = df["cep_usuario"].map(lat_map).astype("float32")
    df["lon_usuario"] = df["cep_usuario"].map(lon_map).astype("float32")
    df["cidade_usuario"] = df["cep_usuario"].map(city_map).astype("string")
    df["uf_usuario"] = df["cep_usuario"].map(region_map).astype("string")

    df["lat_cnes"] = df["cep_cnes_exe"].map(lat_map).astype("float32")
    df["lon_cnes"] = df["cep_cnes_exe"].map(lon_map).astype("float32")
    df["cidade_cnes"] = df["cep_cnes_exe"].map(city_map).astype("string")
    df["uf_cnes"] = df["cep_cnes_exe"].map(region_map).astype("string")

    exact_missing_user = int(df["lat_usuario"].isna().sum())
    exact_missing_cnes = int(df["lat_cnes"].isna().sum())
    log(f"Rows missing user location after exact CEP lookup: {fmt_int(exact_missing_user)}")
    log(f"Rows missing CNES location after exact CEP lookup: {fmt_int(exact_missing_cnes)}")

    df["cep_aprox_usuario"] = np.zeros(len(df), dtype="int8")
    df["cep_aprox_cnes"] = np.zeros(len(df), dtype="int8")
    df["cep_prefix_len_usuario"] = pd.Series(pd.NA, index=df.index, dtype="Int8")
    df["cep_prefix_len_cnes"] = pd.Series(pd.NA, index=df.index, dtype="Int8")

    exact_user_mask = df["lat_usuario"].notna() & df["lon_usuario"].notna()
    exact_cnes_mask = df["lat_cnes"].notna() & df["lon_cnes"].notna()
    df.loc[exact_user_mask, "cep_prefix_len_usuario"] = 8
    df.loc[exact_cnes_mask, "cep_prefix_len_cnes"] = 8

    user_missing_mask = df["lat_usuario"].isna() | df["lon_usuario"].isna()
    cnes_missing_mask = df["lat_cnes"].isna() | df["lon_cnes"].isna()

    recovered_user_n = apply_prefix_recovery(
        df, user_missing_mask, "cep_usuario", "lat_usuario", "lon_usuario",
        "cidade_usuario", "uf_usuario", "cep_aprox_usuario", "cep_prefix_len_usuario",
        prefix_lookup, "user",
    )
    if recovered_user_n == 0:
        log("Recovered user locations by CEP prefix: 0")

    recovered_cnes_n = apply_prefix_recovery(
        df, cnes_missing_mask, "cep_cnes_exe", "lat_cnes", "lon_cnes",
        "cidade_cnes", "uf_cnes", "cep_aprox_cnes", "cep_prefix_len_cnes",
        prefix_lookup, "CNES",
    )
    if recovered_cnes_n == 0:
        log("Recovered CNES locations by CEP prefix: 0")

    # Save unmatched after recovery.
    still_missing_user = df.loc[df["lat_usuario"].isna() | df["lon_usuario"].isna(), "cep_usuario"]
    still_missing_cnes = df.loc[df["lat_cnes"].isna() | df["lon_cnes"].isna(), "cep_cnes_exe"]
    save_series_counts(still_missing_user, OUTPUT_PATHS["cep_usuarios_sem_match_pkl"], OUTPUT_PATHS["cep_usuarios_sem_match_csv"], "cep_usuario")
    save_series_counts(still_missing_cnes, OUTPUT_PATHS["ceps_cnes_sem_match_pkl"], OUTPUT_PATHS["ceps_cnes_sem_match_csv"], "cep_cnes_exe")

    before = len(df)
    location_required = ["lat_usuario", "lon_usuario", "cidade_usuario", "uf_usuario", "lat_cnes", "lon_cnes", "cidade_cnes", "uf_cnes"]
    df = df.dropna(subset=location_required).copy()
    log(f"Dropped {fmt_int(before - len(df))} rows still missing location after prefix recovery.")

    del cep_indexed, lat_map, lon_map, city_map, region_map
    gc.collect()
    return df



In [106]:

# ============================================================================== 
# UF/REGION FEATURES AND DISTANCE
# ==============================================================================


In [107]:
# Add UF codes, macro-regions, and same-city/same-state/same-region indicators comparing user and CNES locations.

def add_uf_region_features(df: pd.DataFrame) -> pd.DataFrame:
    print_section("UF / REGION FEATURES")
    uf = build_uf_lookup()
    uf.to_pickle(OUTPUT_PATHS["ufs_lookup_pkl"])
    uf.to_csv(OUTPUT_PATHS["ufs_lookup_csv"], index=False)
    log(f"Saved UF lookup: {OUTPUT_PATHS['ufs_lookup_pkl']} and {OUTPUT_PATHS['ufs_lookup_csv']}")

    uf_indexed = uf.set_index("sigla")
    cod_map = uf_indexed["codigo_uf"]
    reg_map = uf_indexed["regiao"]

    df["cod_uf_usuario"] = df["uf_usuario"].map(cod_map).astype("Int16")
    df["cod_uf_cnes"] = df["uf_cnes"].map(cod_map).astype("Int16")
    df["regiao_usuario"] = df["uf_usuario"].map(reg_map).astype("string")
    df["regiao_cnes"] = df["uf_cnes"].map(reg_map).astype("string")

    df["mesma_cidade"] = (df["cidade_usuario"] == df["cidade_cnes"]).astype("int8")
    df["mesmo_estado"] = (df["uf_usuario"] == df["uf_cnes"]).astype("int8")
    df["mesma_regiao"] = (df["regiao_usuario"] == df["regiao_cnes"]).astype("int8")
    df.loc[df["regiao_usuario"].isna() | df["regiao_cnes"].isna(), "mesma_regiao"] = 0

    return df



In [108]:
# Create distance_km between user and CNES locations and log descriptive statistics for sanity checking.

def add_distance_feature(df: pd.DataFrame) -> pd.DataFrame:
    print_section("DISTANCE FEATURE")
    df["distance_km"] = calculate_distance_km(
        df["lat_usuario"], df["lon_usuario"], df["lat_cnes"], df["lon_cnes"]
    ).astype("float32")
    log("distance_km created.")
    log(df["distance_km"].describe(percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]).to_string())
    return df



In [109]:

# ==============================================================================
# FINAL CLEANUP, SAVE, AND VALIDATION
# ==============================================================================


In [110]:
# Run final text normalization, audit remaining issues, remove invalid rows under strict rules, convert dtypes, verify the final audit is clean, save the final pickle/sample CSV, and log final class distribution.

def final_cleanup_and_save(df: pd.DataFrame) -> pd.DataFrame:
    print_section("FINAL AUDIT, STRICT CLEANUP, AND SAVE")

    # Normalize text one last time before checking for null-like strings/whitespace.
    # Work column-by-column instead of copying the whole DataFrame.
    text_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
    for col in text_cols:
        df[col] = normalize_text_series(df[col], strip_accents_output=True)
    df = df.replace([np.inf, -np.inf], np.nan)

    audit_and_save(df, "pre_final", OUTPUT_PATHS["quality_pre_final_csv"])

    drop_mask, drop_summary = make_row_drop_report(df)
    drop_summary.to_csv(OUTPUT_PATHS["row_drop_summary_csv"], index=False)
    log(f"Saved row-drop summary: {OUTPUT_PATHS['row_drop_summary_csv']}")
    log(str(drop_summary.sort_values("rows_flagged", ascending=False).head(20)))

    removed_count = int(drop_mask.sum())
    if removed_count > 0:
        # Save only a sample; do not materialize every removed row in RAM.
        df.loc[drop_mask].head(REMOVED_SAMPLE_N).to_csv(OUTPUT_PATHS["final_removed_rows_sample_csv"], index=False)
        log(f"Saved sample of removed rows: {OUTPUT_PATHS['final_removed_rows_sample_csv']}")

    before = len(df)
    df = df.loc[~drop_mask].copy()
    log(f"Final strict cleanup removed {fmt_int(before - len(df))} rows. Remaining: {fmt_int(len(df))}")

    # Dtype convention after all row removals.
    df = convert_final_dtypes(df)

    # Final audit should be clean.
    final_audit = audit_and_save(df, "final_saved", OUTPUT_PATHS["quality_final_csv"])
    unresolved = final_audit[
        (final_audit["missing_count"] > 0)
        | (final_audit["blank_string_count"] > 0)
        | (final_audit["null_like_string_count"] > 0)
        | (final_audit["leading_trailing_whitespace_count"] > 0)
        | (final_audit["infinite_count"] > 0)
    ]
    if not unresolved.empty:
        raise RuntimeError("Final saved dataframe still has unresolved missing/noise values. Check final audit CSV.")

    # Save full pkl and manageable sample CSV.
    df.to_pickle(OUTPUT_PATHS["dados_uteis_pkl"])
    log(f"Saved final refreshed dataset: {OUTPUT_PATHS['dados_uteis_pkl']}")

    sample_n = min(FINAL_SAMPLE_N, len(df))
    df.head(sample_n).to_csv(OUTPUT_PATHS["dados_uteis_sample_csv"], index=False)
    log(f"Saved final sample CSV ({fmt_int(sample_n)} rows): {OUTPUT_PATHS['dados_uteis_sample_csv']}")

    print_class_distribution(df, TARGET_COL, OUTPUT_PATHS["class_distribution_csv"])
    log("\nFinal DataFrame info:")
    df.info(memory_usage="deep")
    return df



In [111]:

# ==============================================================================
# MAIN
# ==============================================================================


In [112]:
# Orchestrate the full preprocessing pipeline: save config, load/clean SISREG, create historical/date features, build CEP resources, enrich locations, add geographic features, compute distance, run final cleanup, and return the final dataset.

def main() -> pd.DataFrame:
    t0 = time.time()
    print_section("REFRESHED DATA PREPARATION PIPELINE")
    log(f"SAVE_TAG: {SAVE_TAG}")
    log("No existing preprocessing files will be overwritten.")
    save_json(CONFIG, OUTPUT_PATHS["pipeline_config_json"])
    log(f"Saved pipeline config: {OUTPUT_PATHS['pipeline_config_json']}")

    df = load_sisreg()
    df = clean_basic_sisreg(df)
    df = add_historical_features(df)
    df = filter_year_and_add_date_parts(df, YEAR_FILTER)

    # Release memory before CEP enrichment.
    gc.collect()

    needed_ceps = build_needed_cep_set(df)
    cep_exact, prefix_lookup = load_and_clean_cep_resources(needed_ceps)
    df = enrich_locations(df, cep_exact, prefix_lookup)
    del needed_ceps, cep_exact, prefix_lookup
    gc.collect()

    df = add_uf_region_features(df)
    df = add_distance_feature(df)
    df = final_cleanup_and_save(df)

    elapsed = (time.time() - t0) / 60
    print_section("PIPELINE COMPLETE")
    log(f"Elapsed time: {elapsed:.2f} minutes")
    log(f"Final output: {OUTPUT_PATHS['dados_uteis_pkl']}")
    return df


if __name__ == "__main__":
    useful_data = main()



REFRESHED DATA PREPARATION PIPELINE
SAVE_TAG: refreshed_v3_memorysafe
No existing preprocessing files will be overwritten.
Saved pipeline config: ../data/processed/pipeline_config_refreshed_v3_memorysafe.json

LOAD RAW SISREG
Loading: ../data/unprocessed/sisreg.csv
Loaded raw shape: (8519214, 15)
Saved quality audit [initial_raw]: ../data/processed/data_quality_audit_initial_raw_refreshed_v3_memorysafe.csv
⚠️ Audit [initial_raw]: found 1 columns with potential quality issues.
        column  missing_count  blank_string_count  null_like_string_count  \
3  cep_usuario              1                   0                       0   

   leading_trailing_whitespace_count  infinite_count  
3                                  0               0  

BASIC SISREG CLEANING
Dropped 0 rows missing cns_usuario or dta_atend.
Dropped 14 rows with invalid CNS/CEP/IBGE merge keys.
Dropped 1.497 rows with invalid idade_usuario/tempo_espera.
Hashed CNS identifiers. Unique patients preserved: 4.523.082
Saved 